In [1]:
import os
from pathlib import Path

import librosa
import numpy as np

project_override = os.environ.get("CARDIOAGENT_PROJECT_ROOT")
if project_override:
    PROJECT_ROOT = Path(project_override).expanduser().resolve()
else:
    PROJECT_ROOT = next(
        (
            candidate.resolve()
            for candidate in (Path.cwd(), *Path.cwd().parents)
            if (candidate / "Data" / "dataset").is_dir()
            and (candidate / "ourdataset" / "train").is_dir()
        ),
        None,
    )

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the CardioAgent-Ops project. Open this notebook from the project, "
        "or set CARDIOAGENT_PROJECT_ROOT to its root directory."
    )

AUDIO_ROOT = PROJECT_ROOT / "ourdataset" / "train"
MEL_ROOT = PROJECT_ROOT / "Data" / "processed" / "features" / "mel_spectrograms"
SPLITS_ROOT = PROJECT_ROOT / "Data" / "processed" / "splits" / "cnn"
MODEL_ROOT = PROJECT_ROOT / "audio_engine" / "artifacts" / "models" / "cnn_custom"
EVALUATION_ROOT = PROJECT_ROOT / "audio_engine" / "evaluation" / "cnn_custom"
REPORT_ROOT = PROJECT_ROOT / "docs" / "report" / "benchmarks" / "cnn_custom"
CLASS_TO_LABEL = {"normal": 0, "abnormal": 1}
FEATURE_VERSION = "mel40_sr2000_gray_v2"

for class_name in CLASS_TO_LABEL:
    class_dir = AUDIO_ROOT / class_name
    if not class_dir.is_dir() or not list(class_dir.glob("*.wav")):
        raise FileNotFoundError(
            f"No WAV files found in {class_dir}. Run the current data-wrangling notebook first."
        )

print(f"Project root: {PROJECT_ROOT}")
print(f"Audio input: {AUDIO_ROOT}")
print(f"Mel features: {MEL_ROOT}")
print(f"Fold manifests: {SPLITS_ROOT}")
print(f"Model output: {MODEL_ROOT}")


Project root: C:\Users\azizb\Downloads\CardioAgent-Ops
Audio input: C:\Users\azizb\Downloads\CardioAgent-Ops\ourdataset\train
Mel features: C:\Users\azizb\Downloads\CardioAgent-Ops\Data\processed\features\mel_spectrograms
Fold manifests: C:\Users\azizb\Downloads\CardioAgent-Ops\Data\processed\splits\cnn
Model output: C:\Users\azizb\Downloads\CardioAgent-Ops\audio_engine\artifacts\models\cnn_custom


In [2]:
import matplotlib.pyplot as plt
from PIL import Image


def make_mel_image(wav_path, output_path):
    signal, sample_rate = librosa.load(wav_path, sr=2000, mono=True)
    if signal.size == 0:
        raise ValueError(f"Audio file is empty: {wav_path}")

    mel = librosa.feature.melspectrogram(
        y=signal,
        sr=sample_rate,
        n_fft=512,
        hop_length=128,
        n_mels=40,
        fmin=20,
        fmax=1000,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    value_range = float(np.ptp(mel_db))
    image = (
        (mel_db - mel_db.min()) / value_range
        if value_range > 0
        else np.zeros_like(mel_db)
    )
    image = np.rint(image * 255).astype(np.uint8)
    Image.fromarray(image).save(output_path)


prepared_counts = {}
for class_name in CLASS_TO_LABEL:
    source_files = sorted((AUDIO_ROOT / class_name).glob("*.wav"))
    output_dir = MEL_ROOT / class_name
    output_dir.mkdir(parents=True, exist_ok=True)
    prepared_count = 0

    for wav_path in source_files:
        image_path = output_dir / f"{wav_path.stem}_{FEATURE_VERSION}.png"
        if image_path.exists() and image_path.stat().st_mtime >= wav_path.stat().st_mtime:
            continue
        make_mel_image(wav_path, image_path)
        prepared_count += 1

    prepared_counts[class_name] = len(list(output_dir.glob(f"*_{FEATURE_VERSION}.png")))
    print(
        f"{class_name}: {prepared_counts[class_name]} spectrograms ready "
        f"({prepared_count} generated this run)"
    )


normal: 1016 spectrograms ready (0 generated this run)
abnormal: 2769 spectrograms ready (0 generated this run)


In [3]:
import csv
import re

from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 3
N_VALIDATION_SPLITS = 5
records = []
for class_name, label in CLASS_TO_LABEL.items():
    image_files = sorted((MEL_ROOT / class_name).glob(f"*_{FEATURE_VERSION}.png"))
    if not image_files:
        raise FileNotFoundError(
            f"No spectrogram PNGs found for {class_name} in {MEL_ROOT / class_name}. "
            "Run the spectrogram-preparation cell first."
        )

    for image_path in image_files:
        feature_suffix = f"_{FEATURE_VERSION}.png"
        if not image_path.name.endswith(feature_suffix):
            raise ValueError(f"Unexpected feature filename: {image_path.name}")
        recording_stem = image_path.name[:-len(feature_suffix)]

        pascal_match = re.match(r"^pascal_(?:normal|murmur|extra)_(\d+)_", recording_stem)
        pascal_recording_match = re.match(
            r"^pascal_(?:normal|murmur|extra)_(\d+)$", recording_stem
        )
        physionet_match = re.match(r"^physio_(training-[a-f])_(.+)$", recording_stem)
        if pascal_match:
            group = f"pascal:patient:{pascal_match.group(1)}"
        elif pascal_recording_match:
            group = f"pascal:recording:{pascal_recording_match.group(1)}"
        elif physionet_match:
            group = f"physionet:{physionet_match.group(1)}:{physionet_match.group(2)}"
        else:
            raise ValueError(
                f"Cannot derive a patient/recording group from {image_path.name}; "
                "update the grouping rule rather than falling back to random splitting."
            )
        records.append((image_path, label, class_name, group))

labels = np.asarray([record[1] for record in records], dtype=np.int32)
groups = np.asarray([record[3] for record in records], dtype=str)
class_counts = np.bincount(labels, minlength=len(CLASS_TO_LABEL))
if int(class_counts.min()) < N_SPLITS:
    raise ValueError(
        f"Each class needs at least {N_SPLITS} images for grouped folds; "
        f"found {dict(zip(CLASS_TO_LABEL, class_counts.tolist()))}."
    )

SPLITS_ROOT.mkdir(parents=True, exist_ok=True)
outer_splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=10
)
for fold_number, (train_val_indices, test_indices) in enumerate(
    outer_splitter.split(np.zeros(len(records)), labels, groups), start=1
):
    inner_splitter = StratifiedGroupKFold(
        n_splits=N_VALIDATION_SPLITS, shuffle=True, random_state=10 + fold_number
    )
    inner_train, validation_relative_indices = next(
        inner_splitter.split(
            np.zeros(len(train_val_indices)),
            labels[train_val_indices],
            groups[train_val_indices],
        )
    )
    train_indices = train_val_indices[inner_train]
    validation_indices = train_val_indices[validation_relative_indices]
    split_indices = {
        "train": train_indices,
        "validation": validation_indices,
        "test": test_indices,
    }

    group_sets = {
        split_name: set(groups[indices]) for split_name, indices in split_indices.items()
    }
    for first, second in (("train", "validation"), ("train", "test"), ("validation", "test")):
        overlap = group_sets[first] & group_sets[second]
        if overlap:
            raise RuntimeError(f"Patient/recording leakage in fold {fold_number}: {overlap}")
    for split_name, indices in split_indices.items():
        split_counts = np.bincount(labels[indices], minlength=len(CLASS_TO_LABEL))
        if np.any(split_counts == 0):
            raise ValueError(
                f"Fold {fold_number} {split_name} split is missing a class: "
                f"{dict(zip(CLASS_TO_LABEL, split_counts.tolist()))}"
            )

    split_path = SPLITS_ROOT / f"cnn_fold_{fold_number}.csv"
    with split_path.open("w", encoding="utf-8", newline="") as split_file:
        writer = csv.DictWriter(
            split_file,
            fieldnames=("split", "relative_path", "label", "class_name", "group"),
        )
        writer.writeheader()
        for split_name, indices in split_indices.items():
            for index in indices:
                image_path, label, class_name, group = records[int(index)]
                writer.writerow(
                    {
                        "split": split_name,
                        "relative_path": image_path.relative_to(PROJECT_ROOT).as_posix(),
                        "label": int(label),
                        "class_name": class_name,
                        "group": group,
                    }
                )
    print(
        f"Fold {fold_number}: {split_path} "
        f"({len(train_indices)} train, {len(validation_indices)} validation, "
        f"{len(test_indices)} test recordings; groups are disjoint)"
    )


Fold 1: C:\Users\azizb\Downloads\CardioAgent-Ops\Data\processed\splits\cnn\cnn_fold_1.csv (2020 train, 504 validation, 1261 test recordings; groups are disjoint)
Fold 2: C:\Users\azizb\Downloads\CardioAgent-Ops\Data\processed\splits\cnn\cnn_fold_2.csv (2018 train, 504 validation, 1263 test recordings; groups are disjoint)
Fold 3: C:\Users\azizb\Downloads\CardioAgent-Ops\Data\processed\splits\cnn\cnn_fold_3.csv (2018 train, 506 validation, 1261 test recordings; groups are disjoint)


In [4]:
import librosa
import sklearn
import tensorflow as tf

print(f"TensorFlow: {tf.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"librosa: {librosa.__version__}")
print("Runtime: local project paths; no Google Drive mount required.")


TensorFlow: 2.21.0
scikit-learn: 1.9.1
librosa: 1.0.1
Runtime: local project paths; no Google Drive mount required.


In [5]:
import csv

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tensorflow.keras import callbacks, layers, models, optimizers

IMAGE_SIZE = (40, 256)
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 0.0001


def load_fold_rows(split_csv, split_name):
    image_paths = []
    labels = []
    groups = []
    with split_csv.open("r", encoding="utf-8", newline="") as split_file:
        for row in csv.DictReader(split_file):
            if row["split"] == split_name:
                image_path = (PROJECT_ROOT / row["relative_path"]).resolve()
                if not image_path.is_file():
                    raise FileNotFoundError(f"CNN input image not found: {image_path}")
                image_paths.append(str(image_path))
                labels.append(int(row["label"]))
                groups.append(row["group"])
    if not image_paths:
        raise ValueError(f"No {split_name} images listed in {split_csv}")
    return image_paths, np.asarray(labels, dtype=np.int32), groups


def mask_spectrogram_axis(image, axis, max_width):
    axis_length = tf.shape(image)[axis]
    mask_width = tf.random.uniform(
        [], minval=0, maxval=max_width + 1, dtype=tf.int32
    )
    start = tf.random.uniform(
        [], minval=0, maxval=axis_length - mask_width + 1, dtype=tf.int32
    )
    positions = tf.range(axis_length)
    keep = tf.cast(
        tf.logical_or(positions < start, positions >= start + mask_width),
        image.dtype,
    )
    if axis == 0:
        mask_shape = tf.stack([axis_length, 1, 1])
    else:
        mask_shape = tf.stack([1, axis_length, 1])
    return image * tf.reshape(keep, mask_shape)


def augment_spectrogram(image):
    image = mask_spectrogram_axis(image, axis=0, max_width=4)
    return mask_spectrogram_axis(image, axis=1, max_width=24)


def make_dataset(image_paths, labels, training, class_weight_vector=None):
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    def decode_png(image_path, label):
        image_bytes = tf.io.read_file(image_path)
        image = tf.io.decode_png(image_bytes, channels=1)
        image = tf.image.resize(image, IMAGE_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    dataset = dataset.map(decode_png, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        if class_weight_vector is None:
            raise ValueError("Training data requires per-fold class weights")
        dataset = dataset.shuffle(
            buffer_size=len(image_paths), seed=10, reshuffle_each_iteration=True
        )
        dataset = dataset.map(
            lambda image, label: (
                augment_spectrogram(image),
                label,
                tf.gather(class_weight_vector, label),
            ),
            num_parallel_calls=tf.data.AUTOTUNE,
        )
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def make_model():
    model = models.Sequential(
        [
            layers.Input(shape=(*IMAGE_SIZE, 1)),
            layers.Conv2D(32, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.Conv2D(32, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.MaxPooling2D(),
            layers.Conv2D(64, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.Conv2D(64, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.MaxPooling2D(),
            layers.Conv2D(128, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.GlobalAveragePooling2D(),
            layers.Dropout(0.35),
            layers.Dense(1, activation="sigmoid"),
        ],
        name="cardio_custom_cnn",
    )
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="roc_auc"),
        ],
    )
    return model


def choose_threshold(labels, probabilities):
    sorted_probabilities = np.sort(np.unique(probabilities))
    midpoints = (
        (sorted_probabilities[:-1] + sorted_probabilities[1:]) / 2
        if len(sorted_probabilities) > 1
        else np.asarray([], dtype=np.float32)
    )
    candidates = np.unique(
        np.concatenate(([0.0, 0.5, 1.0], sorted_probabilities, midpoints))
    )
    return float(
        max(
            candidates,
            key=lambda threshold: (
                balanced_accuracy_score(labels, probabilities >= threshold),
                -abs(float(threshold) - 0.5),
            ),
        )
    )


fold_files = sorted(SPLITS_ROOT.glob("cnn_fold_*.csv"))
if len(fold_files) != N_SPLITS:
    raise FileNotFoundError(
        f"Expected {N_SPLITS} fold CSVs in {SPLITS_ROOT}; found {len(fold_files)}. "
        "Run the grouped split-generation cell first."
    )

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
fold_scores = []

for fold_number, split_csv in enumerate(fold_files, start=1):
    train_paths, train_labels, train_groups = load_fold_rows(split_csv, "train")
    validation_paths, validation_labels, validation_groups = load_fold_rows(
        split_csv, "validation"
    )
    test_paths, test_labels, test_groups = load_fold_rows(split_csv, "test")
    if set(train_groups) & set(validation_groups) or set(train_groups) & set(test_groups) or set(validation_groups) & set(test_groups):
        raise RuntimeError(f"Patient/recording leakage found while loading fold {fold_number}")

    class_counts = np.bincount(train_labels, minlength=len(CLASS_TO_LABEL))
    if np.any(class_counts == 0):
        raise ValueError(f"Training fold {fold_number} is missing a class: {class_counts.tolist()}")
    class_weights = len(train_labels) / (len(CLASS_TO_LABEL) * class_counts.astype(np.float32))
    class_weight_vector = tf.constant(class_weights, dtype=tf.float32)

    train_data = make_dataset(
        train_paths, train_labels, training=True, class_weight_vector=class_weight_vector
    )
    validation_data = make_dataset(validation_paths, validation_labels, training=False)
    test_data = make_dataset(test_paths, test_labels, training=False)

    tf.keras.backend.clear_session()
    model = make_model()
    fold_model_dir = MODEL_ROOT / f"fold_{fold_number}"
    fold_model_dir.mkdir(parents=True, exist_ok=True)
    best_model_path = fold_model_dir / "best.keras"
    checkpoint = callbacks.ModelCheckpoint(
        filepath=str(best_model_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1,
    )
    early_stopping = callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    )
    reduce_learning_rate = callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
    )

    print(
        f"Training fold {fold_number}: {len(train_paths)} train, "
        f"{len(validation_paths)} validation, {len(test_paths)} test"
    )
    history = model.fit(
        train_data,
        validation_data=validation_data,
        epochs=EPOCHS,
        callbacks=[checkpoint, early_stopping, reduce_learning_rate],
        shuffle=False,
        verbose=1,
    )
    best_model = tf.keras.models.load_model(best_model_path)
    validation_probabilities = best_model.predict(validation_data, verbose=0).reshape(-1)
    threshold = choose_threshold(validation_labels, validation_probabilities)
    validation_predictions = (validation_probabilities >= threshold).astype(np.int32)

    test_loss = best_model.evaluate(test_data, return_dict=True, verbose=0)["loss"]
    test_probabilities = best_model.predict(test_data, verbose=0).reshape(-1)
    predictions = (test_probabilities >= threshold).astype(np.int32)
    metrics = {
        "accuracy": float(accuracy_score(test_labels, predictions)),
        "precision": float(precision_score(test_labels, predictions, zero_division=0)),
        "recall": float(recall_score(test_labels, predictions, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(test_labels, predictions)),
        "f1": float(f1_score(test_labels, predictions, zero_division=0)),
        "macro_f1": float(f1_score(test_labels, predictions, average="macro", zero_division=0)),
        "roc_auc": float(roc_auc_score(test_labels, test_probabilities)),
        "test_loss": float(test_loss),
        "validation_balanced_accuracy": float(
            balanced_accuracy_score(validation_labels, validation_predictions)
        ),
        "validation_macro_f1": float(
            f1_score(validation_labels, validation_predictions, average="macro", zero_division=0)
        ),
        "threshold": threshold,
    }
    matrix = confusion_matrix(test_labels, predictions, labels=[0, 1])
    report = classification_report(
        test_labels,
        predictions,
        labels=[0, 1],
        target_names=["normal", "abnormal"],
        zero_division=0,
    )
    fold_scores.append(metrics)

    report_path = EVALUATION_ROOT / f"fold_{fold_number}_report.txt"
    report_path.write_text(
        f"Metrics: {metrics}\nSelected threshold from validation data: {threshold:.4f}\n"
        f"Confusion matrix (normal, abnormal):\n{matrix}\n\n{report}",
        encoding="utf-8",
    )
    figure, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(history.history["accuracy"], label="train")
    axes[0].plot(history.history["val_accuracy"], label="validation")
    axes[0].set_title(f"Fold {fold_number} accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[1].plot(history.history["loss"], label="train")
    axes[1].plot(history.history["val_loss"], label="validation")
    axes[1].set_title(f"Fold {fold_number} loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    figure.tight_layout()
    figure.savefig(REPORT_ROOT / f"fold_{fold_number}_training.png", dpi=150)
    plt.close(figure)

    print(f"Fold {fold_number} metrics: {metrics}")
    print(f"Confusion matrix (normal, abnormal):\n{matrix}")
    print(report)

print("Mean fold accuracy:", float(np.mean([score["accuracy"] for score in fold_scores])))
print(
    "Mean fold balanced accuracy:",
    float(np.mean([score["balanced_accuracy"] for score in fold_scores])),
)
print("Mean abnormal recall:", float(np.mean([score["recall"] for score in fold_scores])))



Training fold 1: 2020 train, 504 validation, 1261 test
Epoch 1/30
126/127 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 0.7207 - loss: 0.5341 - precision: 0.9101 - recall: 0.6861 - roc_auc: 0.8198
Epoch 1: val_loss improved from None to 0.79010, saving model to C:\Users\azizb\Downloads\CardioAgent-Ops\audio_engine\artifacts\models\cnn_custom\fold_1\best.keras

Epoch 1: finished saving model to C:\Users\azizb\Downloads\CardioAgent-Ops\audio_engine\artifacts\models\cnn_custom\fold_1\best.keras
127/127 ━━━━━━━━━━━━━━━━━━━━ 25s 163ms/step - accuracy: 0.7208 - loss: 0.5340 - precision: 0.9102 - recall: 0.6861 - roc_auc: 0.8198 - val_accuracy: 0.2679 - val_loss: 0.7901 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.8329 - learning_rate: 1.0000e-04
Epoch 2/30
126/127 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 0.7793 - loss: 0.4574 - precision: 0.9256 - recall: 0.7593 - roc_auc: 0.8652
Epoch 2: val_loss did not improve from 0.79010
127/127 ━━━━━━━━━━━━━━━━━━━━ 19s 

In [9]:
import os
import re
# ---- MLflow (Service 6) ----
# Cell 4 logs test metrics; best-fold selection uses validation metrics only.
import mlflow

local_tracking_uri = f"sqlite:///{(PROJECT_ROOT / 'mlflow.db').resolve().as_posix()}"
tracking_uri = os.environ.get("MLFLOW_TRACKING_URI") or local_tracking_uri
mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow tracking URI: {tracking_uri}")
mlflow.set_experiment("audio_benchmark")
mlflow.set_system_metrics_sampling_interval(1)
mlflow.enable_system_metrics_logging()

if len(fold_scores) != N_SPLITS:
    raise ValueError(f"Expected {N_SPLITS} fold scores, found {len(fold_scores)}. Run Cell 4 first.")

fold_records = []
for fold_number, scores in enumerate(fold_scores, start=1):
    report_path = EVALUATION_ROOT / f"fold_{fold_number}_report.txt"
    curve_path = REPORT_ROOT / f"fold_{fold_number}_training.png"
    model_path = MODEL_ROOT / f"fold_{fold_number}" / "best.keras"
    for required_path in (report_path, curve_path, model_path):
        if not required_path.is_file():
            raise FileNotFoundError(f"Required fold artifact not found: {required_path}")
    report_text = report_path.read_text(encoding="utf-8")
    matrix_text = report_text.split("Confusion matrix (normal, abnormal):\n", 1)[1].split("\n\n", 1)[0]
    values = [int(value) for value in re.findall(r"\d+", matrix_text)]
    if len(values) != 4:
        raise ValueError(f"Could not read a 2x2 confusion matrix from {report_path}")
    metrics = {
        name: float(scores[name])
        for name in (
            "accuracy", "precision", "recall", "balanced_accuracy", "f1",
            "macro_f1", "roc_auc", "test_loss",
        )
    }
    fold_records.append(
        {
            "fold": fold_number,
            "metrics": metrics,
            "threshold": float(scores["threshold"]),
            "validation_balanced_accuracy": float(scores["validation_balanced_accuracy"]),
            "validation_macro_f1": float(scores["validation_macro_f1"]),
            "matrix": np.asarray(values).reshape(2, 2).tolist(),
            "report_path": report_path,
            "curve_path": curve_path,
            "model_path": model_path,
        }
    )

with mlflow.start_run(run_name="cnn_custom", log_system_metrics=True) as run:
    mlflow.set_tags(
        {
            "model": "custom_cnn",
            "task": "pcg_normal_vs_abnormal",
            "evaluation": "patient_grouped_3_fold_cross_validation",
            "feature_set": FEATURE_VERSION,
        }
    )
    mlflow.log_params(
        {
            "n_folds": N_SPLITS,
            "epochs_requested": EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "input_shape": "40x256x1",
            "mel_bands": 40,
            "sample_rate": 2000,
            "class_weighting": True,
            "spectrogram_masking": True,
        }
    )
    metric_names = (
        "accuracy", "precision", "recall", "balanced_accuracy", "f1", "macro_f1", "roc_auc", "test_loss"
    )
    for metric in metric_names:
        mlflow.log_metric(
            f"mean_{metric}",
            float(np.mean([record["metrics"][metric] for record in fold_records])),
        )
    for record in fold_records:
        for metric, value in record["metrics"].items():
            mlflow.log_metric(f"fold_{metric}", value, step=record["fold"])
        mlflow.log_metric("fold_threshold", record["threshold"], step=record["fold"])
        mlflow.log_metric(
            "fold_validation_balanced_accuracy",
            record["validation_balanced_accuracy"],
            step=record["fold"],
        )
        mlflow.log_artifact(str(record["report_path"]), artifact_path=f"reports/fold_{record['fold']}")
        mlflow.log_artifact(str(record["curve_path"]), artifact_path=f"figures/fold_{record['fold']}")
    mlflow.log_dict(
        {
            "folds": [
                {
                    "fold": record["fold"],
                    **record["metrics"],
                    "threshold": record["threshold"],
                    "validation_balanced_accuracy": record["validation_balanced_accuracy"],
                    "validation_macro_f1": record["validation_macro_f1"],
                }
                for record in fold_records
            ]
        },
        "reports/cv_metrics.json",
    )
    mlflow.log_dict(
        {
            "labels": ["normal", "abnormal"],
            "folds": [
                {"fold": record["fold"], "matrix": record["matrix"]}
                for record in fold_records
            ],
        },
        "reports/confusion_matrices.json",
    )
    best_fold = max(
        fold_records,
        key=lambda record: (
            record["validation_balanced_accuracy"],
            record["validation_macro_f1"],
        ),
    )
    mlflow.set_tag("best_fold", str(best_fold["fold"]))
    mlflow.set_tag("best_threshold", str(best_fold["threshold"]))
    mlflow.log_artifact(str(best_fold["model_path"]), artifact_path="models/best_fold")
    print(f"Logged CNN run: {run.info.run_id}; best fold selected on validation data: {best_fold['fold']}")


MLflow tracking URI: http://localhost:5000


2026/11/16 17:42:43 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/16 17:42:43 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/11/16 17:42:44 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...


Logged CNN run: d0b60123c4bc4488b5cf6e97257c1b89; best fold selected on validation data: 3
🏃 View run cnn_custom at: http://localhost:5000/#/experiments/1/runs/d0b60123c4bc4488b5cf6e97257c1b89
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/11/16 17:42:46 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
